# 05-05 LangGraph 持久化与 Human-in-the-loop

**生产级 Agent 必须**：失败可恢复、可暂停等人审核、多轮记忆、审计追踪。

**本节目标**：Checkpointer 持久化、中断/恢复、时间旅行调试、Human-in-the-loop

---

In [ ]:
import os, sys
sys.path.insert(0, "..")
from dotenv import load_dotenv
load_dotenv("../.env")
from typing import TypedDict

try:
    from langgraph.graph import StateGraph, END
    from langgraph.checkpoint.memory import MemorySaver
    HAS_LG = True
except ImportError:
    HAS_LG = False
    print("LangGraph 未安装")

## 1. MemorySaver（内存持久化）

In [ ]:
class ChatState(TypedDict):
    messages: list
    summary: str

def chatbot(state: ChatState) -> dict:
    from utils.llm_client import call_llm
    msgs = state.get("messages", [])
    last_user = msgs[-1] if msgs else "你好"
    try:
        reply = call_llm(str(last_user), system="你是B站广告助手", max_tokens=100)
    except Exception:
        reply = f"收到: {str(last_user)[:30]}... 我来帮您分析广告数据。"
    return {"messages": msgs + [reply]}

if HAS_LG:
    # 编译时传入 checkpointer
    memory = MemorySaver()
    graph = StateGraph(ChatState)
    graph.add_node("chat", chatbot)
    graph.set_entry_point("chat")
    graph.add_edge("chat", END)
    
    app = graph.compile(checkpointer=memory)
    
    # 使用 thread_id 标识会话
    config = {"configurable": {"thread_id": "user_123"}}
    
    # 第一轮
    r1 = app.invoke({"messages": ["B站广告CTR怎么提升?"], "summary": ""}, config)
    print(f"第1轮: {r1['messages'][-1][:60]}...")
    
    # 第二轮（同一 thread_id，自动恢复上下文）
    r2 = app.invoke({"messages": r1["messages"] + ["具体有哪些优化方法?"], "summary": ""}, config)
    print(f"第2轮: {r2['messages'][-1][:60]}...")
    print(f"\n总消息数: {len(r2['messages'])}（包含历史）")
else:
    print("""
Checkpointer 用法:
  memory = MemorySaver()                      # 内存存储
  app = graph.compile(checkpointer=memory)     # 编译时传入
  config = {"configurable": {"thread_id": "user_123"}}
  app.invoke(state, config)                    # 每次调用自动保存/恢复

Checkpointer 类型:
  MemorySaver       → 内存，重启丢失（开发用）
  SqliteSaver       → SQLite 文件（轻量持久化）
  PostgresSaver     → PostgreSQL（生产环境）
    """)

## 2. Human-in-the-loop（人工审核中断）

In [ ]:
class ReviewState(TypedDict):
    ad_content: str
    ai_review: str
    human_decision: str
    final_status: str

def ai_review(state: ReviewState) -> dict:
    content = state["ad_content"]
    issues = [w for w in ["最", "第一", "绝对"] if w in content]
    result = f"AI审核: {'发现问题: '+','.join(issues) if issues else '未发现明显问题'}"
    print(f"  [AI审核] {result}")
    return {"ai_review": result}

def human_review(state: ReviewState) -> dict:
    # 在生产中这里会暂停等待人类输入
    decision = state.get("human_decision", "approved")
    print(f"  [人工审核] 决定: {decision}")
    return {"final_status": decision}

if HAS_LG:
    review_graph = StateGraph(ReviewState)
    review_graph.add_node("ai_review", ai_review)
    review_graph.add_node("human_review", human_review)
    review_graph.set_entry_point("ai_review")
    review_graph.add_edge("ai_review", "human_review")
    review_graph.add_edge("human_review", END)
    
    # interrupt_before 在 human_review 前暂停
    review_app = review_graph.compile(
        checkpointer=MemorySaver(),
        interrupt_before=["human_review"]
    )
    
    config = {"configurable": {"thread_id": "review_001"}}
    
    # 第一次调用：执行到 human_review 前暂停
    print("=== 步骤1: AI 自动审核 ===")
    r = review_app.invoke(
        {"ad_content": "B站游戏皮肤限时折扣", "ai_review": "", "human_decision": "", "final_status": ""},
        config
    )
    print(f"暂停状态: {r}")
    
    # 模拟人工审核后继续
    print("\n=== 步骤2: 人工审核后继续 ===")
    # 更新 state 中的 human_decision，然后继续执行
    review_app.update_state(config, {"human_decision": "approved"})
    final = review_app.invoke(None, config)  # None = 从中断点继续
    print(f"最终状态: {final.get('final_status', 'N/A')}")
else:
    print("""
Human-in-the-loop 流程:
  1. compile(interrupt_before=["human_review"])  # 设置中断点
  2. app.invoke(state, config)                    # 执行到中断点暂停
  3. state = app.get_state(config)                # 查看当前状态
  4. app.update_state(config, {"human_decision": "approved"})  # 人工修改
  5. app.invoke(None, config)                     # 从中断点继续

B站广告审核场景: AI初审 → 人工复核 → 最终决策
    """)

## 3. 时间旅行调试

In [ ]:
if HAS_LG:
    # 查看状态历史
    print("=== 状态历史（时间旅行）===")
    for i, snapshot in enumerate(review_app.get_state_history(config)):
        print(f"  Checkpoint {i}: {snapshot.values}")
        if i >= 3:
            break
    print("\n可以回滚到任意 checkpoint 重新执行")
else:
    print("""
时间旅行:
  history = app.get_state_history(config)
  for snapshot in history:
      print(snapshot.values)       # 每个时间点的状态
      print(snapshot.config)       # 可用于回滚
  
  # 回滚到特定 checkpoint 重新执行
  app.invoke(None, old_snapshot.config)
    """)

## Checkpointer 对比

| 类型 | 持久化 | 适用场景 | 性能 |
|------|--------|---------|------|
| MemorySaver | 内存（重启丢失） | 开发调试 | 极快 |
| SqliteSaver | SQLite 文件 | 原型/小规模 | 快 |
| PostgresSaver | PostgreSQL | 生产环境 | 中等 |
| RedisSaver | Redis | 高并发生产 | 快 |

## 面试速记

| 问题 | 要点 |
|------|------|
| Checkpointer 的作用 | 保存每步状态，支持恢复/回滚/审计；thread_id 隔离不同用户会话 |
| thread_id 的意义 | 同一 thread_id 共享对话历史，不同 thread_id 完全隔离 |
| interrupt_before vs interrupt_after | before: 节点执行前暂停（可修改输入）；after: 节点执行后暂停（可检查输出） |

**Module 05 完成！** 下一步：`../06-autogen-multi-agent/`